# Logistics Planner Agent

LangGraph-powered pipeline that:
1. **Parses** incoming collection-request emails via GPT-4o
2. **Geocodes** each stop via OpenRouteService (ORS) Pelias API
3. **Optimises** the pickup route via OpenRouteService (ORS) VROOM API
4. **Replies** to the sender with an HTML confirmation email

External tools (not modified here): `gmail_tools.py`, `ors_tools.py`, `sheets_tools.py`, `auth_setup.py`

## 1. Imports & OpenAI Configuration

In [68]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import TypedDict, Optional, List, Annotated, Literal
import logging
import os
import time

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# OpenAI API config
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

## 2. Data Models

In [69]:
class ExtractedEmailContext(BaseModel):
    """One pickup stop extracted from the email."""
    stop_id: str            = Field(description="0-based index as string")
    store_name: str         = Field(description="Store or company name")
    address: str            = Field(description="Full street address")
    time_window: Optional[str]  = Field(None, description="e.g. '09:00-11:00'")
    temperature_control: Optional[bool] = Field(None, description="True if cold-chain required")
    priority: Optional[int] = Field(None, description="1=high, 2=normal, 3=low")
    notes: Optional[str]    = Field(None, description="Special delivery instructions")
    contact_name: Optional[str]   = Field(None, description="Contact person at stop")
    contact_number: Optional[str] = Field(None, description="Contact phone number")
    # Filled after geocoding
    latitude: Optional[float]  = None
    longitude: Optional[float] = None


class OptimizedStop(BaseModel):
    """One stop in the final optimised route."""
    job_id: str
    store_name: str
    address: str
    latitude: float
    longitude: float
    arrival_time_seconds: int       # absolute seconds from midnight (VROOM output)
    service_duration_seconds: int = 300

    @property
    def eta(self) -> str:
        """Format arrival as HH:MM."""
        h, rem = divmod(self.arrival_time_seconds, 3600)
        m = rem // 60
        return f"{h:02d}:{m:02d}"


class RouteResult(BaseModel):
    """Complete optimised route returned by ORS VROOM."""
    total_duration_seconds: int
    total_distance_meters: int
    ordered_stops: List[OptimizedStop]

    @property
    def duration_str(self) -> str:
        h, rem = divmod(self.total_duration_seconds, 3600)
        return f"{h}h {rem // 60}m"

    @property
    def distance_str(self) -> str:
        return f"{self.total_distance_meters / 1000:.1f} km"

## 3. Graph State

In [70]:
class LogisticsState(TypedDict):
    # ── Input ────────────────────────────────────────────────────────────────
    raw_email_content: str
    sender_email: str
    thread_id: str
    # ── Parser node output ───────────────────────────────────────────────────
    parsed_stops: List[dict]
    geocoded_stops: List[dict]
    # ── Optimizer node output ────────────────────────────────────────────────
    route_result: Optional[RouteResult]
    # ── Reply node output ────────────────────────────────────────────────────
    reply_html: str
    # ── Shared ───────────────────────────────────────────────────────────────
    error: Optional[str]
    messages: Annotated[List[BaseMessage], add_messages]


def initial_state(raw_email_content: str, sender_email: str, thread_id: str) -> LogisticsState:
    """Create a clean initial state for each incoming email."""
    return {
        "raw_email_content": raw_email_content,
        "sender_email": sender_email,
        "thread_id": thread_id,
        "parsed_stops": [],
        "geocoded_stops": [],
        "route_result": None,
        "reply_html": "",
        "error": None,
        "messages": [],
    }

## 4. Agent Nodes

Each function is a LangGraph node that receives the full state and returns a **partial** state update.

### Node 1 — Email Parser
Parses the raw email with an LLM, geocodes each stop, and saves both to Google Sheets.

In [71]:
from sheets_tools import save_stops_to_sheet, save_gps_to_sheet
from ors_tools import geocode_address

PARSER_SYSTEM_PROMPT = """\
You are a logistics data extraction assistant.
Parse the email and extract all pickup/collection stops.

For each stop extract:
- stop_id        : 0-based index as string ("0", "1", "2")
- store_name     : name of the store or location
- address        : full street address
- time_window    : optional, e.g. "09:00-11:00"
- temperature_control : true only if cold-chain/refrigeration is explicitly mentioned
- priority       : 1=high, 2=normal, 3=low  (infer from keywords like "urgent", "ASAP")
- notes          : special instructions ("leave at reception", "call on arrival", etc.)
- contact_name   : contact person name if mentioned
- contact_number : phone number if mentioned

Rules:
- Use null for optional fields not mentioned in the email.
- Do not invent information — only extract what is explicitly stated.
"""


class ParsedEmailOutput(BaseModel):
    sender_company: str = Field(description="Company name of the sender")
    stops: List[ExtractedEmailContext] = Field(description="Pickup stops extracted from email")


parser_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
).with_structured_output(ParsedEmailOutput)


def run_parser_agent(state: dict) -> dict:
    raw_email    = state["raw_email_content"]
    thread_id    = state["thread_id"]
    sender_email = state["sender_email"]

    # ── Step 1: LLM extracts structured stops ─────────────────────────────
    response = parser_llm.invoke([
        SystemMessage(content=PARSER_SYSTEM_PROMPT),
        HumanMessage(content=f"Parse this collection request email:\n\n{raw_email}"),
    ])
    stops = [stop.model_dump() for stop in response.stops]

    # ── Step 2: Save raw stops to Sheets ──────────────────────────────────
    save_stops_to_sheet.invoke({"stops": stops, "thread_id": thread_id, "sender_email": sender_email})

    # ── Step 3: Geocode each stop ─────────────────────────────────────────
    geocoded = []
    for stop in stops:
        try:
            geo = geocode_address.invoke({"address": stop["address"]})
            stop["latitude"]   = geo["latitude"]
            stop["longitude"]  = geo["longitude"]
            stop["confidence"] = geo.get("confidence", 0.0)
        except Exception as e:
            log.warning("Geocoding failed for %r: %s", stop["address"], e)
            stop["latitude"]  = None
            stop["longitude"] = None
        geocoded.append(stop)

    # ── Step 4: Save geocoded GPS to Sheets ───────────────────────────────
    valid_geocoded = [s for s in geocoded if s.get("latitude") is not None]
    if valid_geocoded:
        save_gps_to_sheet.invoke({"geocoded_stops": valid_geocoded})

    log.info("Parsed %d stops, geocoded %d successfully.", len(stops), len(valid_geocoded))
    return {"parsed_stops": stops, "geocoded_stops": geocoded}

### Node 2 — Route Optimizer
Sends geocoded stops to ORS VROOM and builds the ordered `RouteResult`.

In [72]:
from ors_tools import optimize_route
from sheets_tools import save_route_to_sheet


def run_route_optimizer(state: dict) -> dict:
    geocoded_stops = state.get("geocoded_stops", [])
    valid_stops = [
        s for s in geocoded_stops
        if s.get("latitude") is not None and s.get("longitude") is not None
    ]

    if len(valid_stops) < 2:
        return {"error": f"Need at least 2 geocoded stops for optimization. Got {len(valid_stops)}."}

    # ── Step 1: Format payload for ORS VROOM ──────────────────────────────
    stops_payload = [
        {
            "stop_index": i,
            "store_name": s["store_name"],
            "address": s["address"],
            "latitude": s["latitude"],
            "longitude": s["longitude"],
        }
        for i, s in enumerate(valid_stops)
    ]

    # ── Step 2: Call ORS optimization endpoint ────────────────────────────
    raw_result = optimize_route.invoke({"stops": stops_payload})

    # ── Step 3: Build domain model ────────────────────────────────────────
    ordered_stops = [
        OptimizedStop(
            job_id=str(s["job_id"]),
            store_name=s["store_name"],
            address=s["address"],
            latitude=s["latitude"],
            longitude=s["longitude"],
            arrival_time_seconds=s["arrival_time_seconds"],
            service_duration_seconds=s.get("service_duration_seconds", 300),
        )
        for s in raw_result["ordered_stops"]
    ]

    route_result = RouteResult(
        total_duration_seconds=raw_result["total_duration_seconds"],
        total_distance_meters=raw_result["total_distance_meters"],
        ordered_stops=ordered_stops,
    )

    # ── Step 4: Save to Sheets ────────────────────────────────────────────
    save_route_to_sheet.invoke({
        "route_result": {
            "total_duration_seconds": route_result.total_duration_seconds,
            "total_distance_meters": route_result.total_distance_meters,
            "ordered_stops": [s.model_dump() for s in route_result.ordered_stops],
        }
    })

    log.info("Route optimised: %s, %s, %d stops.",
             route_result.duration_str, route_result.distance_str, len(ordered_stops))
    return {"route_result": route_result}

### Node 3 — Reply Agent
Uses an LLM to compose a professional HTML confirmation email and sends it via Gmail.

In [73]:
from gmail_tools import send_gmail_reply

REPLY_SYSTEM_PROMPT = """\
You are a logistics operations assistant for a circular economy transportation company.

Compose a professional HTML email confirming a multi-stop reusable-packaging collection route.

The email must:
1. Open with a friendly greeting using the sender's company name.
2. Confirm the collection request has been received and the route is optimised.
3. Include an HTML table with columns: Stop # | Store Name | Address | Estimated Arrival
4. State total driving time and total distance.
5. Close with a support contact placeholder.
6. Use clean inline CSS — no external stylesheets.
7. Be written in English.

Output ONLY raw HTML — no markdown, no explanation, no code fences.
"""

reply_llm = ChatOpenAI(
    model="gpt-4o",
    api_key=OPENAI_API_KEY,
    temperature=0.3,
)


def _format_route_for_prompt(route_result: RouteResult) -> str:
    lines = [
        f"Total driving time : {route_result.duration_str}",
        f"Total distance     : {route_result.distance_str}",
        "",
        "Ordered stop sequence:",
    ]
    for i, stop in enumerate(route_result.ordered_stops, 1):
        lines.append(f"  {i}. {stop.store_name} | {stop.address} | ETA: {stop.eta}")
    return "\n".join(lines)


def run_reply_agent(state: dict) -> dict:
    route_result: RouteResult = state.get("route_result")
    thread_id    = state.get("thread_id", "")
    sender_email = state.get("sender_email", "")
    raw_email    = state.get("raw_email_content", "")

    if not sender_email:
        return {"error": "sender_email is empty — cannot send reply"}

    # ── Step 1: Build LLM prompt ──────────────────────────────────────────
    route_summary = _format_route_for_prompt(route_result)

    user_prompt = (
        f"Sender's original request:\n---\n{raw_email}\n---\n\n"
        f"Optimised route:\n{route_summary}\n\n"
        f"Reply to: {sender_email}\n\n"
        "Compose the HTML confirmation email now."
    )

    # ── Step 2: LLM generates HTML email ─────────────────────────────────
    response = reply_llm.invoke([
        SystemMessage(content=REPLY_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])
    html_body = response.content.strip()

    # Strip accidental code fences
    if html_body.startswith("```"):
        parts = html_body.split("```")
        html_body = parts[1].lstrip("html").strip() if len(parts) > 1 else html_body

    # ── Step 3: Send Gmail reply ──────────────────────────────────────────
    send_result = send_gmail_reply.invoke({
        "thread_id": thread_id,
        "to": sender_email,
        "subject": "Collection Route Confirmation",
        "html_body": html_body,
    })

    log.info("Reply sent to %s: %s", sender_email, send_result)
    return {
        "reply_html": html_body,
        "messages": [HumanMessage(content=f"Reply sent: {send_result}")],
    }

## 5. Graph Definition

```
START
  └─► parser_agent
          ├─► route_optimizer  (≥2 geocoded stops)
          │       ├─► reply_agent  ──► END
          │       └─► error_handler ──► END
          └─► error_handler ──► END
```

In [74]:
def route_after_parse(state: dict) -> Literal["route_optimizer", "error_handler"]:
    """Proceed to optimizer only if we have ≥2 valid geocoded stops."""
    if state.get("error"):
        return "error_handler"
    valid = [s for s in state.get("geocoded_stops", []) if s.get("latitude")]
    return "route_optimizer" if len(valid) >= 2 else "error_handler"


def route_after_optimize(state: dict) -> Literal["reply_agent", "error_handler"]:
    """Proceed to reply only if optimization succeeded."""
    if state.get("error") or state.get("route_result") is None:
        return "error_handler"
    return "reply_agent"


def handle_error(state: dict) -> dict:
    """Log workflow failures and surface them in the final state."""
    error_msg = state.get("error", "Unknown error")
    log.error("Workflow failed for thread %s: %s", state.get("thread_id"), error_msg)
    return {"error": error_msg}


def build_graph():
    builder = StateGraph(LogisticsState)

    # Register nodes
    builder.add_node("parser_agent",   run_parser_agent)
    builder.add_node("route_optimizer", run_route_optimizer)
    builder.add_node("reply_agent",    run_reply_agent)
    builder.add_node("error_handler",  handle_error)

    # Edges
    builder.add_edge(START, "parser_agent")
    builder.add_conditional_edges("parser_agent",   route_after_parse)
    builder.add_conditional_edges("route_optimizer", route_after_optimize)
    builder.add_edge("reply_agent",   END)
    builder.add_edge("error_handler", END)

    app = builder.compile(checkpointer=MemorySaver())
    log.info("LangGraph compiled successfully.")
    return app


app = build_graph()

2026-04-08 21:16:16,731 [INFO] LangGraph compiled successfully.


## 6. Entry Point

`process_email` runs the graph for a single email.  
`main` polls Gmail in a loop and dispatches each new email.

In [75]:
from gmail_tools import poll_gmail_inbox

POLL_INTERVAL = int(os.getenv("GMAIL_POLL_INTERVAL", "60"))
GMAIL_QUERY   = os.getenv("GMAIL_QUERY", "is:unread subject:collection request")


def process_email(app, raw_email: str, thread_id: str, sender_email: str) -> dict:
    """Run the full logistics workflow for a single email."""
    config = {"configurable": {"thread_id": thread_id}}
    state  = initial_state(raw_email, sender_email, thread_id)

    log.info("Processing email from %s (thread: %s)", sender_email, thread_id)
    result = app.invoke(state, config=config)

    if result.get("error"):
        log.error("Workflow failed: %s", result["error"])
    else:
        log.info("Workflow complete. Reply sent to %s", sender_email)

    return result


def main():
    """Poll Gmail and dispatch new collection-request emails."""
    log.info("Starting Gmail polling every %ds. Query: %r", POLL_INTERVAL, GMAIL_QUERY)
    processed_ids: set = set()

    while True:
        try:
            emails = poll_gmail_inbox.invoke({"query": GMAIL_QUERY})
            new_emails = [e for e in emails if e["message_id"] not in processed_ids]

            if new_emails:
                log.info("Found %d new email(s).", len(new_emails))

            for email in new_emails:
                processed_ids.add(email["message_id"])
                process_email(
                    app=app,
                    raw_email=email["body"],
                    thread_id=email["thread_id"],
                    sender_email=email["sender_email"],
                )

        except Exception as exc:
            log.exception("Polling error: %s", exc)

        time.sleep(POLL_INTERVAL)



main()

2026-04-08 21:16:16,742 [INFO] Starting Gmail polling every 60s. Query: 'is:unread subject:Pickup Schedule'
2026-04-08 21:17:20,109 [INFO] Found 1 new email(s).
2026-04-08 21:17:20,112 [INFO] Processing email from Anuj Mumbaikar <anuj.mumbaikar05@gmail.com> (thread: 19d6dc5e7f0ffdb6)
2026-04-08 21:17:30,503 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:17:41,811 [INFO] Parsed 3 stops, geocoded 3 successfully.
2026-04-08 21:17:47,083 [INFO] ORS optimization response: {'code': 0, 'summary': {'cost': 2623, 'routes': 1, 'unassigned': 0, 'delivery': [0], 'amount': [0], 'pickup': [0], 'setup': 0, 'service': 900, 'duration': 2623, 'waiting_time': 0, 'priority': 0, 'violations': [], 'computing_times': {'loading': 4000, 'solving': 1, 'routing': 0}}, 'unassigned': [], 'routes': [{'vehicle': 1, 'cost': 2623, 'delivery': [0], 'amount': [0], 'pickup': [0], 'setup': 0, 'service': 900, 'duration': 2623, 'waiting_time': 0, 'priority': 0, 'steps': 

KeyboardInterrupt: 